# Step 5.4 — Inter-Annotator Agreement (Round 5 Canonical Schema)

Compares R3 and R8 independent blind annotations on the shared 14-window round 1 sample.

**Inputs:**
- `data/annotations/r3_manual_annotations_round1_canonical.csv` (R3, Round 5 canonical)
- `data/annotations/r8_manual_annotations_round1.csv` (R8, Round 5 canonical)

**Shared windows:** 14

**Results summary:**

| Schema | Dimension | Kappa | Level |
|---|---|---|---|
| 5.1-A | `friction_type` | 0.8293 | Almost perfect |
| 5.1-A | `severity_s` (nominal) | 0.3378 | Fair |
| 5.1-A | `severity_s` (weighted) | 0.6056 | Substantial |
| 5.1-A | `sentiment_e` | 0.2222 | Fair |
| 5.1-A | `calibrator_score_l` (nominal) | 0.8971 | Almost perfect |
| 5.1-A | `calibrator_score_l` (weighted) | 0.9271 | Almost perfect |
| 5.1-A | `signal_alignment` | N/A | 100% agree (single class) |
| 5.1-B | `narration_quality` | 0.5882 | Moderate |
| 5.1-B | `recording_quality` | 0.0 (92.9% agree) | Class imbalance |
| 5.1-B | `coaching_evidence` | N/A | 100% agree (single class) |

**Eval discipline:**
- `sentiment_e = E3` (neutral expressed) ≠ null (no verbal expression)
- `calibrator_score_l` is audit signal only; not merged with `severity_s`
- Disagreement `window_id` lists are output per dimension for Step 5.4 error analysis


In [14]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')
print('Working directory:', os.getcwd())


Working directory: c:\Users\Sialiang\OneDrive - The University of Sydney (Students)\capstone\seemeplease\notebooks


## 1. Load Annotation Files

In [15]:
R3_PATH = 'data/annotations/r3_manual_annotations_round1_canonical.csv'
R8_PATH = 'data/annotations/r8_manual_annotations_round1.csv'

r3 = pd.read_csv(R3_PATH, encoding='latin-1')
r8 = pd.read_csv(R8_PATH, encoding='latin-1')

print(f'R3 rows: {len(r3)}, columns: {list(r3.columns)}')
print(f'R8 rows: {len(r8)}, columns: {list(r8.columns)}')


R3 rows: 14, columns: ['window_id', 'project', 'video_id', 'task_title', 'text', 'task_instructions', 'annotated', 'finding', 'observed_signal', 'stated_signal', 'signal_alignment', 'friction_type', 'severity_s', 'sentiment_e', 'calibrator_score_l', 'rationale', 'structural_amplification_note', 'narration_quality', 'recording_quality', 'coaching_evidence', 'confidence', 'notes', 'annotator']
R8 rows: 14, columns: ['window_id', 'project', 'video_id', 'task_title', 'text', 'task_instructions', 'annotated', 'finding', 'observed_signal', 'stated_signal', 'signal_alignment', 'friction_type', 'severity_s', 'sentiment_e', 'calibrator_score_l', 'rationale', 'structural_amplification_note', 'narration_quality', 'recording_quality', 'coaching_evidence', 'confidence', 'notes', 'annotator']


## 2. Align on Shared Windows

In [16]:
shared_ids = sorted(set(r3['window_id']) & set(r8['window_id']))
print(f'Shared windows: {len(shared_ids)}')
print(shared_ids)

r3s = r3[r3['window_id'].isin(shared_ids)].set_index('window_id').loc[shared_ids]
r8s = r8[r8['window_id'].isin(shared_ids)].set_index('window_id').loc[shared_ids]


Shared windows: 14
['Sharelinsonny_uq_w026', 'fjone7_uq_w066', 'gameoverdan_suncorp_w040', 'ghum_wa_w029', 'giuliaclemente26_uq_w004', 'giuliaclemente26_uq_w050', 'margieflint_suncorp_w007', 'marychaunguyen_suncorp_w011', 'oliviamitchell22_suncorp_w007', 'oliviamitchell22_suncorp_w017', 'ramazankawish_wa_w075', 'reneerussell99_uq_w009', 'thanoptions_uq_w008', 'tianarosie1_wa_w015']


## 3. Helper Functions

In [17]:
def compute_kappa(r3_col, r8_col, dim, weights=None):
    df = pd.DataFrame({'r3': r3_col, 'r8': r8_col}).dropna()
    df = df[(df['r3'].astype(str).str.strip() != '') &
            (df['r8'].astype(str).str.strip() != '')]
    if len(df) < 2:
        print(f'{dim}: insufficient data ({len(df)} rows)')
        return None, pd.DataFrame()
    try:
        k = cohen_kappa_score(df['r3'], df['r8'], weights=weights)
        label = f'(weighted={weights})' if weights else ''
        print(f'{dim:35s} {label:20s} kappa = {k:.4f}  (n={len(df)})')
        return round(k, 4), df
    except Exception as e:
        print(f'{dim}: error — {e}')
        return None, df

def crosstab_and_disagree(r3_col, r8_col, dim):
    df = pd.DataFrame({'R3': r3_col, 'R8': r8_col}).dropna()
    df = df[(df['R3'].astype(str).str.strip() != '') &
            (df['R8'].astype(str).str.strip() != '')]
    if df.empty:
        return
    ct = pd.crosstab(df['R3'], df['R8'], margins=True)
    print(f'\n--- {dim} cross-tab ---')
    display(ct)
    disagree = df[df['R3'] != df['R8']]
    if not disagree.empty:
        print(f'Disagreements ({len(disagree)} windows):')
        display(disagree)
    else:
        print('No disagreements.')


## 4. 5.1-A Finding-Level Kappa

Five dimensions: `friction_type` / `severity_s` / `sentiment_e` / `calibrator_score_l` / `signal_alignment`

Note: `signal_alignment` kappa will show `nan` if only one class is present in the sample — this is expected and does not indicate an error.


In [18]:
print('=' * 65)
print('5.1-A FINDING-LEVEL KAPPA')
print('=' * 65)

# friction_type
k_friction, _ = compute_kappa(r3s['friction_type'], r8s['friction_type'], 'friction_type')

# severity_s — nominal
k_sev, _ = compute_kappa(r3s['severity_s'], r8s['severity_s'], 'severity_s')

# severity_s — linear weighted (ordinal S1>S2>...>S6>none>unclear)
SEV_ORDER = {'S1':0,'S2':1,'S3':2,'S4':3,'S5':4,'S6':5,'none':6,'unclear':7}
def sev_weighted(r3_col, r8_col):
    df = pd.DataFrame({'r3':r3_col,'r8':r8_col}).dropna()
    df = df[(df['r3'].astype(str).str.strip()!='')&(df['r8'].astype(str).str.strip()!='')]
    df['r3o'] = df['r3'].map(SEV_ORDER)
    df['r8o'] = df['r8'].map(SEV_ORDER)
    df = df.dropna(subset=['r3o','r8o'])
    if len(df)<2: return None
    k = cohen_kappa_score(df['r3o'].astype(int), df['r8o'].astype(int), weights='linear')
    print(f'{'severity_s':35s} {'(weighted=linear)':20s} kappa = {k:.4f}  (n={len(df)})')
    return round(k,4)
k_sev_w = sev_weighted(r3s['severity_s'], r8s['severity_s'])

# sentiment_e — treat blank/NaN as 'null'
r3_sent = r3s['sentiment_e'].fillna('null').astype(str).str.strip().replace('','null')
r8_sent = r8s['sentiment_e'].fillna('null').astype(str).str.strip().replace('','null')
k_sent, _ = compute_kappa(r3_sent, r8_sent, 'sentiment_e')

# calibrator_score_l — nominal
k_cal, _ = compute_kappa(r3s['calibrator_score_l'], r8s['calibrator_score_l'], 'calibrator_score_l')

# calibrator_score_l — linear weighted (ordinal L1-L5)
CAL_ORDER = {'L1':0,'L2':1,'L3':2,'L4':3,'L5':4}
def cal_weighted(r3_col, r8_col):
    df = pd.DataFrame({'r3':r3_col,'r8':r8_col}).dropna()
    df = df[(df['r3'].astype(str).str.strip()!='')&(df['r8'].astype(str).str.strip()!='')]
    df['r3o'] = df['r3'].map(CAL_ORDER)
    df['r8o'] = df['r8'].map(CAL_ORDER)
    df = df.dropna(subset=['r3o','r8o'])
    if len(df)<2: return None
    k = cohen_kappa_score(df['r3o'].astype(int), df['r8o'].astype(int), weights='linear')
    print(f'{'calibrator_score_l':35s} {'(weighted=linear)':20s} kappa = {k:.4f}  (n={len(df)})')
    return round(k,4)
k_cal_w = cal_weighted(r3s['calibrator_score_l'], r8s['calibrator_score_l'])

# signal_alignment
k_align, _ = compute_kappa(r3s['signal_alignment'], r8s['signal_alignment'], 'signal_alignment')


5.1-A FINDING-LEVEL KAPPA
friction_type                                            kappa = 0.8293  (n=14)
severity_s                                               kappa = 0.3378  (n=14)
severity_s                          (weighted=linear)    kappa = 0.6056  (n=14)
sentiment_e                                              kappa = 0.2222  (n=14)
calibrator_score_l                                       kappa = 0.8971  (n=14)
calibrator_score_l                  (weighted=linear)    kappa = 0.9271  (n=14)
signal_alignment                                         kappa = nan  (n=14)


## 5. 5.1-B Video-Level Kappa

Three dimensions: `narration_quality` / `recording_quality` / `coaching_evidence`

> Video-level labels should be consistent within the same `video_id`. Kappa is computed at window level; note the smaller effective sample size.

Note: `coaching_evidence` kappa will show `nan` if all windows share the same label (e.g. all `none`) — this is expected. `recording_quality` kappa = 0.0 when one class dominates (13/14 = `good`); observed agreement is 92.9% and the kappa value is not meaningful here.


In [19]:
print('=' * 65)
print('5.1-B VIDEO-LEVEL KAPPA')
print('=' * 65)

k_narr,  _ = compute_kappa(r3s['narration_quality'],  r8s['narration_quality'],  'narration_quality')
k_rec,   _ = compute_kappa(r3s['recording_quality'],  r8s['recording_quality'],  'recording_quality')
k_coach, _ = compute_kappa(r3s['coaching_evidence'],  r8s['coaching_evidence'],  'coaching_evidence')


5.1-B VIDEO-LEVEL KAPPA
narration_quality                                        kappa = 0.5882  (n=14)
recording_quality                                        kappa = 0.0000  (n=14)
coaching_evidence                                        kappa = nan  (n=14)


## 6. Cross-Tab Agreement Tables & Disagreement Window Lists

In [20]:
print('5.1-A Cross-tabs')
crosstab_and_disagree(r3s['friction_type'],      r8s['friction_type'],      'friction_type')
crosstab_and_disagree(r3s['severity_s'],          r8s['severity_s'],          'severity_s')
crosstab_and_disagree(r3_sent,                    r8_sent,                    'sentiment_e')
crosstab_and_disagree(r3s['calibrator_score_l'],  r8s['calibrator_score_l'],  'calibrator_score_l')
crosstab_and_disagree(r3s['signal_alignment'],    r8s['signal_alignment'],    'signal_alignment')


5.1-A Cross-tabs

--- friction_type cross-tab ---


R8,F1,F2,F3,F5,F6,F7,none,All
R3,,,,,,,,
F1,2,0,0,0,0,0,0,2
F2,0,2,0,1,0,0,0,3
F3,0,0,2,0,0,0,0,2
F6,0,0,0,0,2,0,0,2
F7,0,0,0,0,0,1,1,2
none,0,0,0,0,0,0,3,3
All,2,2,2,1,2,1,4,14


Disagreements (2 windows):


,R3,R8
window_id,,
giuliaclemente26_uq_w004,F2,F5
tianarosie1_wa_w015,F7,none



--- severity_s cross-tab ---


...

Disagreements (1 windows):


,R3,R8
window_id,,
tianarosie1_wa_w015,L2,L1



--- signal_alignment cross-tab ---


R8,aligned,All
R3,,
aligned,14,14
All,14,14


No disagreements.


In [21]:
print('5.1-B Cross-tabs')
crosstab_and_disagree(r3s['narration_quality'], r8s['narration_quality'], 'narration_quality')
crosstab_and_disagree(r3s['recording_quality'], r8s['recording_quality'], 'recording_quality')
crosstab_and_disagree(r3s['coaching_evidence'], r8s['coaching_evidence'], 'coaching_evidence')


5.1-B Cross-tabs

--- narration_quality cross-tab ---


R8,adequate,rich,sparse,All
R3,,,,
adequate,4,1,1,6
rich,1,7,0,8
All,5,8,1,14


Disagreements (3 windows):


,R3,R8
window_id,,
gameoverdan_suncorp_w040,adequate,rich
reneerussell99_uq_w009,rich,adequate
tianarosie1_wa_w015,adequate,sparse



--- recording_quality cross-tab ---


...

Disagreements (1 windows):


,R3,R8
window_id,,
ghum_wa_w029,good,acceptable



--- coaching_evidence cross-tab ---


R8,none,All
R3,,
none,14,14
All,14,14


No disagreements.


## 7. Summary Table

In [22]:
summary = pd.DataFrame([
    {'schema':'5.1-A','dimension':'friction_type',           'kappa':k_friction, 'note':'nominal'},
    {'schema':'5.1-A','dimension':'severity_s',              'kappa':k_sev,      'note':'nominal'},
    {'schema':'5.1-A','dimension':'severity_s (weighted)',   'kappa':k_sev_w,    'note':'linear weighted, ordinal S1>S6>none'},
    {'schema':'5.1-A','dimension':'sentiment_e',             'kappa':k_sent,     'note':'null = no verbal expression'},
    {'schema':'5.1-A','dimension':'calibrator_score_l',      'kappa':k_cal,      'note':'audit signal only'},
    {'schema':'5.1-A','dimension':'calibrator_score_l (weighted)','kappa':k_cal_w,'note':'linear weighted, ordinal L1>L5'},
    {'schema':'5.1-A','dimension':'signal_alignment',        'kappa':k_align,    'note':'aligned/conflicted/stated_missing'},
    {'schema':'5.1-B','dimension':'narration_quality',       'kappa':k_narr,     'note':'video-level, small n'},
    {'schema':'5.1-B','dimension':'recording_quality',       'kappa':k_rec,      'note':'video-level, small n'},
    {'schema':'5.1-B','dimension':'coaching_evidence',       'kappa':k_coach,    'note':'video-level, small n'},
])
summary['kappa'] = summary['kappa'].apply(lambda x: round(x,4) if x is not None else 'N/A')
print('\n' + '='*65)
print('KAPPA SUMMARY — Round 5 Canonical Schema')
print('='*65)
display(summary)



KAPPA SUMMARY — Round 5 Canonical Schema


,schema,dimension,kappa,note
0,5.1-A,friction_type,0.8293,nominal
1,5.1-A,severity_s,0.3378,nominal
2,5.1-A,severity_s (weighted),0.6056,"linear weighted, ordinal S1>S6>none"
3,5.1-A,sentiment_e,0.2222,null = no verbal expression
4,5.1-A,calibrator_score_l,0.8971,audit signal only
5,5.1-A,calibrator_score_l (weighted),0.9271,"linear weighted, ordinal L1>L5"
6,5.1-A,signal_alignment,NaN,aligned/conflicted/stated_missing
7,5.1-B,narration_quality,0.5882,"video-level, small n"
8,5.1-B,recording_quality,0.0000,"video-level, small n"
9,5.1-B,coaching_evidence,NaN,"video-level, small n"


## 8. Supporting Fields — Presence Audit

Per Step 5.3: `observed_signal` / `stated_signal` / `rationale` / `structural_amplification_note` are audited for presence (not Kappa).

Expected results based on annotation instructions:
- `observed_signal`, `stated_signal`, `rationale`: 100% present for both annotators
- `structural_amplification_note`: filled only when a cohort-specific amplification applies; most windows will be blank (2/14 expected — `ghum_wa_w029` PDF accessibility case and `marychaunguyen_suncorp_w011` VoiceOver case)


In [23]:
support_cols = ['observed_signal','stated_signal','rationale','structural_amplification_note']
print('Supporting field presence (% non-blank):')
print(f'  {'Field':40s}  R3      R8')
for col in support_cols:
    if col not in r3s.columns or col not in r8s.columns:
        print(f'  {col:40s}  (column not present)')
        continue
    r3p = (r3s[col].notna() & (r3s[col].astype(str).str.strip()!='')).mean()
    r8p = (r8s[col].notna() & (r8s[col].astype(str).str.strip()!='')).mean()
    print(f'  {col:40s}  {r3p*100:.0f}%     {r8p*100:.0f}%')


Supporting field presence (% non-blank):
  Field                                     R3      R8
  observed_signal                           100%     100%
  stated_signal                             100%     100%
  rationale                                 100%     100%
  structural_amplification_note             14%     14%


## 9. Interpretation

Cohen's kappa measures inter-annotator agreement beyond chance.

**Kappa interpretation (Landis & Koch 1977):**

| Range | Interpretation |
|---|---|
| < 0.20 | Slight |
| 0.21–0.40 | Fair |
| 0.41–0.60 | Moderate |
| 0.61–0.80 | Substantial |
| 0.81–1.00 | Almost perfect |

---

### 9.1 5.1-A Finding-Level Results

**`friction_type` — kappa = 0.8293 (Almost perfect)**
Strongest agreement in the 5.1-A dimensions. 2 disagreements out of 14 windows:
- `giuliaclemente26_uq_w004`: R3=F2, R8=F5. Both annotators identified interface friction but disagreed on whether the trigger was user uncertainty (F2 Confidence) or unexpected behaviour (F5 Unexpected Behaviour). Borderline case where the prompt interruption is the primary trigger.
- `tianarosie1_wa_w015`: R3=F7, R8=none. R3 classified as Excessive Effort; R8 judged no friction. This is a sensitive content page where user silence was interpreted differently — R3 read it as cognitive load, R8 as no impediment.

**`severity_s` nominal — kappa = 0.3378 (Fair); weighted — kappa = 0.6056 (Substantial)**
The large gap between nominal (0.3378) and weighted (0.6056) confirms that all 7 disagreements are between adjacent severity levels — S3↔S2, S3↔S4, S5↔S4, S5↔S6, S5↔none. No large jumps (e.g. S1↔S6) were observed. The weighted kappa of 0.6056 is the more meaningful measure and indicates substantial agreement when ordinal distance is considered.

**`sentiment_e` — kappa = 0.2222 (Fair)**
Lowest agreement among 5.1-A dimensions. 8 disagreements, all concentrated on the E3/E4 boundary:
- R3 consistently labelled 6 windows as E4 (Negative/Frustrated) that R8 labelled E3 (Neutral)
- 1 additional disagreement: `oliviamitchell22_suncorp_w007` R3=E4, R8=E2 (positive)
- 1 disagreement: `oliviamitchell22_suncorp_w017` R3=E1, R8=E2 (both positive, adjacent)
The dominant pattern is the E3/E4 boundary — R3 applied a lower threshold for frustration than R8. This is the hardest boundary to calibrate in the sentiment framework.
**→ Recommend: joint E3/E4 calibration session on 3–4 borderline examples before Step 5.4 final evaluation.**

**`calibrator_score_l` nominal — kappa = 0.8971; weighted — kappa = 0.9271 (Almost perfect)**
Highest agreement overall. Only 1 disagreement (`tianarosie1_wa_w015`: R3=L2, R8=L1, adjacent levels). The L-scale gate definitions appear to be the most consistently applied dimension across both annotators.

**`signal_alignment` — kappa = N/A (100% observed agreement)**
All 14 windows labelled `aligned` by both annotators. Kappa is undefined when only one class is present in the sample. This is a valid outcome — all 14 sampled windows had both observed and stated signals. The `conflicted` and `stated_missing` values may appear more frequently in the full 55-video dataset.

---

### 9.2 5.1-B Video-Level Results

**`narration_quality` — kappa = 0.5882 (Moderate)**
3 disagreements out of 14 windows:
- `gameoverdan_suncorp_w040`: R3=adequate, R8=rich
- `reneerussell99_uq_w009`: R3=rich, R8=adequate
- `tianarosie1_wa_w015`: R3=adequate, R8=sparse (window with mostly silent reading)
All disagreements are between adjacent levels. Moderate agreement is acceptable for a 4-level session-level assessment.

**`recording_quality` — kappa = 0.0 (misleading due to class imbalance)**
13 of 14 windows agree on `good`; observed agreement = 92.9%. Kappa = 0.0 because when one class dominates, expected agreement ≈ observed agreement, making kappa uninformative. 1 disagreement: `ghum_wa_w029` (R3=good, R8=acceptable — the PDF accessibility session). This kappa value should NOT be interpreted as poor agreement.

**`coaching_evidence` — kappa = N/A (100% observed agreement)**
All 14 windows labelled `none` by both annotators. Kappa undefined (single class). Consistent with the development sample — no moderator coaching was detected in any window.

---

### 9.3 Supporting Fields

| Field | R3 present | R8 present | Notes |
|---|---|---|---|
| `observed_signal` | 100% | 100% | ✅ Both annotators filled all windows |
| `stated_signal` | 100% | 100% | ✅ Both annotators filled all windows |
| `rationale` | 100% | 100% | ✅ Both annotators filled all windows |
| `structural_amplification_note` | 14% | 14% | ✅ 2/14 windows — correct (ghum_wa_w029 + marychaunguyen_suncorp_w011) |

---

### 9.4 Summary & Action Items

| Dimension | Kappa | Level | Action |
|---|---|---|---|
| `friction_type` | 0.8293 | Almost perfect | ✅ No action needed |
| `severity_s` (weighted) | 0.6056 | Substantial | ✅ Acceptable; adjacent-level gap is expected |
| `sentiment_e` | 0.2222 | Fair | ⚠️ Joint E3/E4 calibration session recommended |
| `calibrator_score_l` (weighted) | 0.9271 | Almost perfect | ✅ No action needed |
| `signal_alignment` | N/A | 100% agree | ✅ No action needed |
| `narration_quality` | 0.5882 | Moderate | ✅ Acceptable for session-level assessment |
| `recording_quality` | 0.0 (92.9% agree) | Class imbalance | ✅ Near-unanimous; kappa not meaningful |
| `coaching_evidence` | N/A | 100% agree | ✅ No action needed |

**Eval discipline reminders:**
- `calibrator_score_l` is an audit signal only — do NOT merge with `severity_s` in downstream evaluation.
- `sentiment_e = E3` (neutral expressed) ≠ null (no verbal expression). The 6 E3/E4 disagreements are annotation calibration gaps, not schema violations.
- 5.1-B kappa values should be interpreted with caution given the small effective sample (multiple windows per video share the same label).
- Disagreement `window_id` lists in Section 6 are the primary input for Step 5.4 error analysis.


## 10. LLM V2 vs R8 round-1

Compares Layer 3 LLM V2 outputs against R8's manual annotations on the same 14 round-1 windows.

**Inputs:**
- `data/processed/layer3_findings_filtered.csv` — LLM V2 findings (pseudo-positives removed)
- `data/processed/layer3_video_assessments.csv` — LLM V2 video-level labels
- `data/annotations/r8_manual_annotations_round1.csv` — R8 human ground truth

**Multi-finding windows (4 windows with ≥2 LLM findings):**
`ramazankawish_wa_w075`, `marychaunguyen_suncorp_w011`, `giuliaclemente26_uq_w004`, `reneerussell99_uq_w009`.
Primary selection rule: highest `calibrator_score_l`; ties broken by first occurrence (row order).

**No-finding windows (4 windows absent from LLM output):**
`tianarosie1_wa_w015`, `thanoptions_uq_w008`, `fjone7_uq_w066`, `oliviamitchell22_suncorp_w017`.
Treated as `friction_type='none'` / `severity_s='none'` for finding-level Kappa.
`sentiment_e` and `calibrator_score_l` are excluded from Kappa for these windows (LLM produces no value).

**Decision gate (from Step 5.4 spec):** `friction_type` κ ≥ 0.5 → V2 acceptable, no V3 revision needed.

In [24]:
LLM_FINDINGS_PATH = 'data/processed/layer3_findings_filtered.csv'
LLM_VIDEO_PATH    = 'data/processed/layer3_video_assessments.csv'

llm_findings = pd.read_csv(LLM_FINDINGS_PATH, encoding='utf-8')
llm_video    = pd.read_csv(LLM_VIDEO_PATH,    encoding='utf-8')
r8           = pd.read_csv(R8_PATH, encoding='latin-1')

r8_window_ids = r8['window_id'].tolist()

# Deduplicate: keep the primary (highest calibrator_score_l) LLM finding per window.
# Ties broken by first occurrence (row order preserved).
def pick_primary(grp):
    grp = grp.copy()
    grp['_rank'] = grp['calibrator_score_l'].map(CAL_ORDER).fillna(-1)
    return grp.nlargest(1, '_rank', keep='first').drop(columns='_rank')

llm_14      = llm_findings[llm_findings['window_id'].isin(r8_window_ids)].copy()
llm_primary = (llm_14
               .groupby('window_id', sort=False, group_keys=False)
               .apply(pick_primary)
               .reset_index(drop=True))

has_llm = set(llm_primary['window_id'])
no_llm  = [w for w in r8_window_ids if w not in has_llm]
print(f'Windows with LLM finding:    {len(has_llm)}')
print(f'Windows without LLM finding: {len(no_llm)}: {no_llm}')

# --- Merge R8 + LLM finding-level ---
r8_idx  = r8.set_index('window_id').copy()
llm_idx = llm_primary.set_index('window_id')[
    ['friction_type', 'severity_s', 'sentiment_e', 'calibrator_score_l']
].copy()
llm_idx.columns = ['llm_ft', 'llm_sev', 'llm_sent', 'llm_cal']

cmp = r8_idx.join(llm_idx, how='left')
cmp['llm_ft']  = cmp['llm_ft'].fillna('none')   # no-finding window → none
cmp['llm_sev'] = cmp['llm_sev'].fillna('none')  # no-finding window → none
# llm_sent / llm_cal remain NaN for no-finding windows (excluded from Kappa below)

# Add video-level LLM labels (joined via video_id)
llm_vid = llm_video.set_index('video_id')[['narration_quality', 'recording_quality']].copy()
llm_vid.columns = ['llm_narr', 'llm_rec']
cmp = cmp.join(llm_vid, on='video_id', how='left')

print('\nR8 vs LLM comparison table (14 windows):')
display(cmp[['friction_type', 'llm_ft',
             'severity_s',    'llm_sev',
             'sentiment_e',   'llm_sent',
             'calibrator_score_l', 'llm_cal',
             'narration_quality',  'llm_narr',
             'recording_quality',  'llm_rec']]
        .rename(columns={
            'friction_type':       'R8_ft',   'llm_ft':   'LLM_ft',
            'severity_s':          'R8_sev',  'llm_sev':  'LLM_sev',
            'sentiment_e':         'R8_sent', 'llm_sent': 'LLM_sent',
            'calibrator_score_l':  'R8_cal',  'llm_cal':  'LLM_cal',
            'narration_quality':   'R8_narr', 'llm_narr': 'LLM_narr',
            'recording_quality':   'R8_rec',  'llm_rec':  'LLM_rec',
        }))

# ─── 6-Dimension Kappa ───────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('LLM V2 vs R8 — 6-DIMENSION KAPPA')
print('=' * 65)

# 1. friction_type  (n=14; no-finding windows treat as 'none')
k_ft_llm, _  = compute_kappa(cmp['friction_type'], cmp['llm_ft'],
                               'friction_type')

# 2. severity_s nominal  (n=14)
k_sev_llm, _ = compute_kappa(cmp['severity_s'], cmp['llm_sev'],
                               'severity_s (nominal)')

# 3. severity_s weighted  (n=14)
k_sev_w_llm  = sev_weighted(cmp['severity_s'], cmp['llm_sev'])

# 4. sentiment_e  (n=10; no-finding windows excluded — LLM has no sentiment)
sent_mask     = cmp['llm_sent'].notna()
k_sent_llm, _ = compute_kappa(
    cmp.loc[sent_mask, 'sentiment_e'],
    cmp.loc[sent_mask, 'llm_sent'],
    f'sentiment_e (n={sent_mask.sum()}, finding windows only)')

# 5. calibrator_score_l  (n=10; no-finding windows excluded)
cal_mask      = cmp['llm_cal'].notna()
k_cal_llm, _  = compute_kappa(
    cmp.loc[cal_mask, 'calibrator_score_l'],
    cmp.loc[cal_mask, 'llm_cal'],
    f'calibrator_score_l (n={cal_mask.sum()})')
k_cal_w_llm   = cal_weighted(
    cmp.loc[cal_mask, 'calibrator_score_l'],
    cmp.loc[cal_mask, 'llm_cal'])

# 6. narration_quality  (n=14; video-level)
k_narr_llm, _ = compute_kappa(cmp['narration_quality'], cmp['llm_narr'],
                                'narration_quality (video-level)')

# 7. recording_quality  (n=14; video-level)
k_rec_llm, _  = compute_kappa(cmp['recording_quality'], cmp['llm_rec'],
                                'recording_quality (video-level)')

# 8. signal_alignment — structural note
# R8 labels all windows 'aligned'; LLM labels no-finding windows 'stated_missing'
# This is a schema convention difference, not a quality issue
r8_align_llm  = cmp['signal_alignment'].fillna('').astype(str)
llm_align_col = cmp['llm_ft'].apply(lambda x: 'stated_missing' if x == 'none' else 'aligned')
k_align_llm, _ = compute_kappa(r8_align_llm, llm_align_col, 'signal_alignment (schema convention diff)')

# ─── Summary table ────────────────────────────────────────────────────────────
def fmt(v):
    if v is None: return 'N/A'
    try:
        return f'{float(v):.4f}'
    except (TypeError, ValueError):
        return 'N/A'

summary_llm = pd.DataFrame([
    {'dimension': 'friction_type',
     'kappa_R3vsR8': 0.8293, 'kappa_LLMvsR8': k_ft_llm,
     'note': 'nominal, n=14'},
    {'dimension': 'severity_s (nominal)',
     'kappa_R3vsR8': 0.3378, 'kappa_LLMvsR8': k_sev_llm,
     'note': 'nominal, n=14'},
    {'dimension': 'severity_s (weighted)',
     'kappa_R3vsR8': 0.6056, 'kappa_LLMvsR8': k_sev_w_llm,
     'note': 'linear weighted ordinal'},
    {'dimension': 'sentiment_e',
     'kappa_R3vsR8': 0.2222, 'kappa_LLMvsR8': k_sent_llm,
     'note': 'finding windows only, n=10'},
    {'dimension': 'calibrator_score_l',
     'kappa_R3vsR8': 0.8971, 'kappa_LLMvsR8': k_cal_llm,
     'note': 'finding windows only, n=10'},
    {'dimension': 'calibrator_score_l (weighted)',
     'kappa_R3vsR8': 0.9271, 'kappa_LLMvsR8': k_cal_w_llm,
     'note': 'linear weighted ordinal'},
    {'dimension': 'narration_quality',
     'kappa_R3vsR8': 0.5882, 'kappa_LLMvsR8': k_narr_llm,
     'note': 'video-level, n=14; LLM defaults to "rich"'},
    {'dimension': 'recording_quality',
     'kappa_R3vsR8': 0.0,    'kappa_LLMvsR8': k_rec_llm,
     'note': 'video-level, n=14; LLM defaults to "acceptable"'},
    {'dimension': 'signal_alignment',
     'kappa_R3vsR8': 'N/A',  'kappa_LLMvsR8': k_align_llm,
     'note': 'schema convention diff: R8=aligned, LLM=stated_missing for no-finding windows'},
])
summary_llm['kappa_R3vsR8']  = summary_llm['kappa_R3vsR8'].apply(fmt)
summary_llm['kappa_LLMvsR8'] = summary_llm['kappa_LLMvsR8'].apply(fmt)

pd.set_option('display.max_colwidth', None)
print('\n' + '=' * 65)
print('KAPPA SUMMARY — LLM V2 vs R8  (R3 vs R8 shown for reference)')
print('=' * 65)
display(summary_llm)

Windows with LLM finding:    10
Windows without LLM finding: 4: ['tianarosie1_wa_w015', 'thanoptions_uq_w008', 'fjone7_uq_w066', 'oliviamitchell22_suncorp_w017']

R8 vs LLM comparison table (14 windows):


,R8_ft,LLM_ft,R8_sev,LLM_sev,R8_sent,LLM_sent,R8_cal,LLM_cal,R8_narr,LLM_narr,R8_rec,LLM_rec
window_id,,,,,,,,,,,,
ramazankawish_wa_w075,F6,F6,S4,S4,E4,E4,L3,L3,adequate,rich,good,acceptable
gameoverdan_suncorp_w040,F6,F6,S5,S4,E3,E4,L2,L3,rich,rich,good,acceptable
Sharelinsonny_uq_w026,F1,F1,S6,S5,E3,E4,L2,L2,rich,rich,good,acceptable
oliviamitchell22_suncorp_w007,F1,F1,S6,S5,E2,E4,L1,L2,rich,rich,good,acceptable
giuliaclemente26_uq_w050,F7,F1,S4,S5,E4,E4,L3,L2,rich,rich,good,acceptable
reneerussell99_uq_w009,F2,F2,S5,S5,E3,E4,L2,L2,adequate,rich,good,good
ghum_wa_w029,F3,F3,S2,S3,E4,E4,L4,L3,adequate,rich,acceptable,acceptable
marychaunguyen_suncorp_w011,F3,F7,S4,S4,E3,E4,L3,L3,rich,rich,good,acceptable
giuliaclemente26_uq_w004,F5,F7,S5,S5,E4,E4,L2,L2,rich,rich,good,acceptable



LLM V2 vs R8 — 6-DIMENSION KAPPA
friction_type                                            kappa = 0.7407  (n=14)
severity_s (nominal)                                     kappa = 0.5238  (n=14)
severity_s                          (weighted=linear)    kappa = 0.7603  (n=14)
sentiment_e (n=10, finding windows only)                      kappa = 0.0000  (n=10)
calibrator_score_l (n=10)                                kappa = 0.3103  (n=10)
calibrator_score_l                  (weighted=linear)    kappa = 0.4118  (n=10)
narration_quality (video-level)                          kappa = 0.0000  (n=14)
recording_quality (video-level)                          kappa = 0.0118  (n=14)
signal_alignment (schema convention diff)                      kappa = 0.0000  (n=14)

KAPPA SUMMARY — LLM V2 vs R8  (R3 vs R8 shown for reference)


,dimension,kappa_R3vsR8,kappa_LLMvsR8,note
0,friction_type,0.8293,0.7407,"nominal, n=14"
1,severity_s (nominal),0.3378,0.5238,"nominal, n=14"
2,severity_s (weighted),0.6056,0.7603,linear weighted ordinal
3,sentiment_e,0.2222,0.0000,"finding windows only, n=10"
4,calibrator_score_l,0.8971,0.3103,"finding windows only, n=10"
5,calibrator_score_l (weighted),0.9271,0.4118,linear weighted ordinal
6,narration_quality,0.5882,0.0000,"video-level, n=14; LLM defaults to ""rich"""
7,recording_quality,0.0000,0.0118,"video-level, n=14; LLM defaults to ""acceptable"""
8,signal_alignment,N/A,0.0000,"schema convention diff: R8=aligned, LLM=stated_missing for no-finding windows"


In [25]:
print('=' * 65)
print('ERROR CASES — LOW-KAPPA DIMENSIONS (LLM vs R8, 5-10 cases)')
print('=' * 65)

# (A) sentiment_e — expected lowest kappa; systematic E3→E4 pattern
print('\n--- (A) sentiment_e disagreements ---')
print('Pattern: LLM inflates neutral (E3) → negative/frustrated (E4)')
sent_err = (cmp.loc[sent_mask & (cmp['sentiment_e'] != cmp['llm_sent']),
                    ['sentiment_e', 'llm_sent', 'friction_type', 'severity_s', 'stated_signal']]
              .rename(columns={'sentiment_e': 'R8',   'llm_sent': 'LLM',
                               'friction_type': 'ft', 'severity_s': 'sev'}))
display(sent_err)

# (B) friction_type — disagreements (expected ~3 cases)
print('\n--- (B) friction_type disagreements ---')
ft_err = (cmp.loc[cmp['friction_type'] != cmp['llm_ft'],
                  ['friction_type', 'llm_ft', 'severity_s', 'llm_sev', 'stated_signal']]
            .rename(columns={'friction_type': 'R8', 'llm_ft': 'LLM',
                             'severity_s': 'R8_sev', 'llm_sev': 'LLM_sev'}))
display(ft_err)

# (C) narration_quality — LLM defaults to "rich" for all videos
print('\n--- (C) narration_quality disagreements (LLM = "rich" for all 14) ---')
narr_err = (cmp.loc[cmp['narration_quality'] != cmp['llm_narr'],
                    ['narration_quality', 'llm_narr', 'video_id']]
              .rename(columns={'narration_quality': 'R8', 'llm_narr': 'LLM'}))
display(narr_err)

# (D) recording_quality — LLM defaults to "acceptable", R8 says "good" for most
print(f'\n--- (D) recording_quality: {(cmp["recording_quality"] != cmp["llm_rec"]).sum()}/14 disagree ---')
print('Pattern: LLM uses "acceptable" for all videos; R8 uses "good" for most.')
print('Systematic label bias — not a case-by-case disagreement.')

ERROR CASES — LOW-KAPPA DIMENSIONS (LLM vs R8, 5-10 cases)

--- (A) sentiment_e disagreements ---
Pattern: LLM inflates neutral (E3) → negative/frustrated (E4)


,R8,LLM,ft,sev,stated_signal
window_id,,,,,
gameoverdan_suncorp_w040,E3,E4,F6,S5,"We tried this before, that wasn't it."
Sharelinsonny_uq_w026,E3,E4,F1,S6,"Written in formal legal language, it's not very user-friendly, and I wouldn't realistically read all of it in detail."
oliviamitchell22_suncorp_w007,E2,E4,F1,S6,What's third party? I have no clue.
reneerussell99_uq_w009,E3,E4,F2,S5,"I didn't see that. It's right up the top. I was expecting that to be where it's asking like, are you an Australian citizen."
marychaunguyen_suncorp_w011,E3,E4,F3,S4,"For someone using voiceover, there's a lot to listen to. I'm just going line by line at the moment."
margieflint_suncorp_w007,E3,E4,F2,S5,I don't know if I trust them. I've never heard of them. I still don't see NRMA on there.



--- (B) friction_type disagreements ---


,R8,LLM,R8_sev,LLM_sev,stated_signal
window_id,,,,,
giuliaclemente26_uq_w050,F7,F1,S4,S5,I need to install something which is already a pain. How is a person going to input something like this and remember where they were?
marychaunguyen_suncorp_w011,F3,F7,S4,S4,"For someone using voiceover, there's a lot to listen to. I'm just going line by line at the moment."
giuliaclemente26_uq_w004,F5,F7,S5,S5,The thing that really bothers me is when they try and have you download things or input things that you haven't even had the time to look at the landing page. Get off my back.



--- (C) narration_quality disagreements (LLM = "rich" for all 14) ---


,R8,LLM,video_id
window_id,,,
ramazankawish_wa_w075,adequate,rich,ramazankawish_wa
reneerussell99_uq_w009,adequate,rich,reneerussell99_uq
ghum_wa_w029,adequate,rich,ghum_wa
margieflint_suncorp_w007,adequate,rich,margieflint_suncorp
tianarosie1_wa_w015,sparse,rich,tianarosie1_wa
thanoptions_uq_w008,adequate,rich,thanoptions_uq



--- (D) recording_quality: 12/14 disagree ---
Pattern: LLM uses "acceptable" for all videos; R8 uses "good" for most.
Systematic label bias — not a case-by-case disagreement.


### 10.6 Conclusion

**Final judge: `friction_type` κ = 0.7407 (Substantial) ≥ 0.5 → V2 prompt passes the gate. No V3 revision needed.**

| Dimension | LLM V2 vs R8 κ | Level | Verdict |
|---|---|---|---|
| `friction_type` | **0.7407** | Substantial | ✅ Gate passed (≥ 0.5) |
| `severity_s` (weighted) | **0.7603** | Substantial | ✅ Strong ordinal agreement |
| `severity_s` (nominal) | **0.5238** | Moderate | ✅ Acceptable |
| `sentiment_e` | **0.0000** | Slight | ⚠️ Systematic E3→E4 inflation by LLM |
| `calibrator_score_l` (weighted) | **0.4118** | Fair | ⚠️ Moderate gap; no-finding windows excluded |
| `signal_alignment` | schema diff | — | ℹ️ R8=aligned vs LLM=stated_missing for no-finding windows |
| `narration_quality` | **0.0000** | Slight | ⚠️ LLM defaults to `rich` for all videos |
| `recording_quality` | **0.0118** | Slight | ⚠️ LLM defaults to `acceptable`; R8 uses `good` for most |
| `coaching_evidence` | N/A | — | ✅ All `none` on both sides |

**Key findings:**

- **`friction_type`** agreement is substantial (κ = 0.7407) with only 3 disagreements — `giuliaclemente26_uq_w050` (R8=F7, LLM=F1), `marychaunguyen_suncorp_w011` (R8=F3, LLM=F7), and `giuliaclemente26_uq_w004` (R8=F5, LLM=F7). All are borderline multi-friction windows where the dominant friction type is genuinely ambiguous.

- **`sentiment_e`** κ = 0.0 reflects a systematic LLM bias: V2 assigns E4 (Frustrated) to almost all windows regardless of actual tester tone. R8 correctly differentiates E2/E3/E4. This does not trigger a V3 revision but should be addressed with E3 calibration examples if sentiment accuracy becomes a downstream requirement.

- **`calibrator_score_l`** gap (κ = 0.3103 nominal) is partly structural — the 4 no-finding windows have no LLM score, reducing effective n to 10. Weighted kappa (0.4118) is more informative and shows fair directional agreement.

- **Video-level (5.1-B) labels** show systematic default behaviour: LLM assigns `rich` to all `narration_quality` and `acceptable` to all `recording_quality`, regardless of session content. This is a V2 prompt calibration gap in the 5.1-B assessment, separate from the finding-level friction verdict.

**One-sentence conclusion:**  
> LLM V2 `friction_type` κ = **0.7407** (Substantial) — V2 prompt is retained; `sentiment_e` (κ = 0.0) and 5.1-B video-level labels show systematic calibration gaps that do not require a V3 prompt revision but should be noted for downstream use.
